In [ ]:
import os
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

In [ ]:
DATASET_PATH = r"sampled_xiangqi_dataset\balanced_value_dataset_abs_buckets_v1_tanh15.npz"
OUTPUT_DIR = Path("value_training_runs_huber_single_npz")
OUTPUT_DIR.mkdir(exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "best_value_model_huber.pth"
LAST_MODEL_PATH = OUTPUT_DIR / "last_value_model_huber.pth"
HISTORY_CSV_PATH = OUTPUT_DIR / "training_history_huber.csv"

# dataset keys
X_KEY = "X"
Y_KEY = "y"

# tensor shape
INPUT_CHANNELS = 15
BOARD_H = 10
BOARD_W = 9

# split
SEED = 42
VAL_RATIO = 0.10

# runtime
BATCH_SIZE = 256
NUM_WORKERS = 0
USE_AMP = torch.cuda.is_available()

# weighted sampling on train split
USE_WEIGHTED_SAMPLER = True
WEIGHT_BINS = np.array(
    [0, 1, 2, 3, 5, 7.5, 10, 12.5, 15.0001],
    dtype=np.float32
)
WEIGHT_POWER = 0.5
MAX_WEIGHT_MULTIPLIER = 4.0

# model
TRUNK_CHANNELS = 96
NUM_BLOCKS = 6
DROPOUT = 0.05
HEAD_HIDDEN = 128

# train
NUM_EPOCHS = 80
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
HUBER_BETA = 1.0
GRAD_CLIP_NORM = 1.0

# scheduler
LR_FACTOR = 0.5
LR_PATIENCE = 3
MIN_LR = 1e-6

# early stopping
USE_EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 8

# metrics
# accuracy = fraction with |pred-target| <= tolerance
ACC_TOLERANCE = 1.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

class ValueDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def print_target_stats(name, y):
    print(f"\n=== {name} ===")
    print("size:", len(y))
    print("min :", float(np.min(y)))
    print("max :", float(np.max(y)))
    print("mean:", float(np.mean(y)))
    print("std :", float(np.std(y)))

    abs_y = np.abs(y)
    print("abs 0-1     :", int(((abs_y >= 0) & (abs_y < 1)).sum()))
    print("abs 1-2     :", int(((abs_y >= 1) & (abs_y < 2)).sum()))
    print("abs 2-3     :", int(((abs_y >= 2) & (abs_y < 3)).sum()))
    print("abs 3-5     :", int(((abs_y >= 3) & (abs_y < 5)).sum()))
    print("abs 5-7.5   :", int(((abs_y >= 5) & (abs_y < 7.5)).sum()))
    print("abs 7.5-10  :", int(((abs_y >= 7.5) & (abs_y < 10)).sum()))
    print("abs 10-12.5 :", int(((abs_y >= 10) & (abs_y < 12.5)).sum()))
    print("abs 12.5-15 :", int(((abs_y >= 12.5) & (abs_y <= 15)).sum()))


def make_stratified_split_indices(y, val_ratio=0.10, seed=42):
    """
    Stratify by |y| bucket so train/val keep a similar target shape.
    """
    rng = np.random.default_rng(seed)
    abs_y = np.abs(y)
    bin_ids = np.digitize(abs_y, WEIGHT_BINS, right=False)

    train_indices = []
    val_indices = []

    for b in np.unique(bin_ids):
        idx = np.where(bin_ids == b)[0]
        rng.shuffle(idx)

        n_val = max(1, int(len(idx) * val_ratio)) if len(idx) > 1 else 0
        val_idx = idx[:n_val]
        train_idx = idx[n_val:]

        val_indices.append(val_idx)
        train_indices.append(train_idx)

    train_indices = np.concatenate(train_indices) if len(train_indices) > 0 else np.array([], dtype=np.int64)
    val_indices = np.concatenate(val_indices) if len(val_indices) > 0 else np.array([], dtype=np.int64)

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    return train_indices, val_indices


def compute_sample_weights(y):
    abs_y = np.abs(np.asarray(y, dtype=np.float32))
    bin_ids = np.digitize(abs_y, WEIGHT_BINS, right=False)

    unique_bins, counts = np.unique(bin_ids, return_counts=True)
    count_map = {b: c for b, c in zip(unique_bins, counts)}

    weights = []
    for b in bin_ids:
        cnt = count_map.get(b, 1)
        w = 1.0 / (float(cnt) ** WEIGHT_POWER)
        weights.append(w)

    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / max(weights.mean(), 1e-12)
    weights = np.clip(weights, 1.0 / MAX_WEIGHT_MULTIPLIER, MAX_WEIGHT_MULTIPLIER)
    return weights

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, channels, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = out + identity
        out = self.relu(out)
        return out


class ValueResNet(nn.Module):
    def __init__(self, in_channels=15, trunk_channels=96, num_blocks=6, dropout=0.05, head_hidden=128):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, trunk_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(trunk_channels),
            nn.ReLU(inplace=True),
        )

        self.blocks = nn.Sequential(*[
            BasicBlock(trunk_channels, dropout=dropout) for _ in range(num_blocks)
        ])

        # lighter head than full flatten -> less overfitting
        self.head = nn.Sequential(
            nn.Conv2d(trunk_channels, trunk_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(trunk_channels),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(trunk_channels, head_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout) if dropout > 0 else nn.Identity(),
            nn.Linear(head_hidden, 1),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        return x.squeeze(1)


def value_accuracy(preds, targets, tolerance=1.0):
    return (torch.abs(preds - targets) <= tolerance).float().mean().item()

def batch_mae(preds, targets):
    return torch.abs(preds - targets).mean().item()

def batch_rmse(preds, targets):
    return torch.sqrt(torch.mean((preds - targets) ** 2)).item()

def get_lr(optimizer):
    return optimizer.param_groups[0]["lr"]


def train_one_epoch(model, loader, optimizer, criterion, scaler=None):
    model.train()

    total_loss = 0.0
    total_acc = 0.0
    total_mae = 0.0
    total_rmse = 0.0
    total_samples = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if USE_AMP:
            with torch.cuda.amp.autocast():
                preds = model(xb)
                loss = criterion(preds, yb)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        total_acc += value_accuracy(preds.detach(), yb, tolerance=ACC_TOLERANCE) * bs
        total_mae += batch_mae(preds.detach(), yb) * bs
        total_rmse += batch_rmse(preds.detach(), yb) * bs
        total_samples += bs

    return {
        "loss": total_loss / total_samples,
        "acc": total_acc / total_samples,
        "mae": total_mae / total_samples,
        "rmse": total_rmse / total_samples,
    }


@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_mae = 0.0
    total_rmse = 0.0
    total_samples = 0

    all_preds = []
    all_targets = []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        preds = model(xb)
        loss = criterion(preds, yb)

        bs = xb.size(0)
        total_loss += loss.item() * bs
        total_acc += value_accuracy(preds, yb, tolerance=ACC_TOLERANCE) * bs
        total_mae += batch_mae(preds, yb) * bs
        total_rmse += batch_rmse(preds, yb) * bs
        total_samples += bs

        all_preds.append(preds.cpu().numpy())
        all_targets.append(yb.cpu().numpy())

    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    return {
        "loss": total_loss / total_samples,
        "acc": total_acc / total_samples,
        "mae": total_mae / total_samples,
        "rmse": total_rmse / total_samples,
        "preds": all_preds,
        "targets": all_targets,
    }



In [ ]:
data = np.load(DATASET_PATH, allow_pickle=True)

if X_KEY not in data or Y_KEY not in data:
    raise ValueError(f"Expected keys '{X_KEY}' and '{Y_KEY}', got {list(data.keys())}")

X = data[X_KEY].astype(np.float32)
y = data[Y_KEY].astype(np.float32)

assert X.ndim == 4, f"Expected X [N,C,H,W], got {X.shape}"
assert X.shape[1] == INPUT_CHANNELS, f"Expected {INPUT_CHANNELS} channels, got {X.shape[1]}"
assert X.shape[2] == BOARD_H and X.shape[3] == BOARD_W, f"Expected {BOARD_H}x{BOARD_W}, got {X.shape[2:]}"
assert y.ndim == 1, f"Expected y [N], got {y.shape}"
assert len(X) == len(y), "X and y length mismatch"

print("\nLoaded dataset:", DATASET_PATH)
print("X shape:", X.shape)
print("y shape:", y.shape)

train_idx, val_idx = make_stratified_split_indices(y, val_ratio=VAL_RATIO, seed=SEED)

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]

print_target_stats("Train target stats", y_train)
print_target_stats("Val target stats", y_val)

train_ds = ValueDataset(X_train, y_train)
val_ds = ValueDataset(X_val, y_val)

if USE_WEIGHTED_SAMPLER:
    train_weights = compute_sample_weights(y_train)
    print("\ntrain weight min/max:", float(train_weights.min()), float(train_weights.max()))

    train_sampler = WeightedRandomSampler(
        weights=torch.as_tensor(train_weights, dtype=torch.double),
        num_samples=len(train_weights),
        replacement=True
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )
else:
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\ntrain batches:", len(train_loader))
print("val batches  :", len(val_loader))

In [ ]:
model = ValueResNet(
    in_channels=INPUT_CHANNELS,
    trunk_channels=TRUNK_CHANNELS,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
    head_hidden=HEAD_HIDDEN
).to(device)

criterion = nn.SmoothL1Loss(beta=HUBER_BETA)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=LR_FACTOR,
    patience=LR_PATIENCE,
    min_lr=MIN_LR,
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("\nModel created:")
print(model)
print("criterion:", criterion)
print("initial lr:", get_lr(optimizer))

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, scaler=scaler)
    val_metrics = eval_one_epoch(model, val_loader, criterion)

    scheduler.step(val_metrics["loss"])

    row = {
        "epoch": epoch,
        "lr": get_lr(optimizer),

        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "train_mae": train_metrics["mae"],
        "train_rmse": train_metrics["rmse"],

        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_mae": val_metrics["mae"],
        "val_rmse": val_metrics["rmse"],
    }
    history.append(row)

    improved = row["val_loss"] < best_val_loss

    print(
        f"Epoch [{epoch:03d}/{NUM_EPOCHS:03d}] | "
        f"lr={row['lr']:.6f} | "
        f"train_loss={row['train_loss']:.6f} | "
        f"train_acc={row['train_acc']:.4f} | "
        f"train_mae={row['train_mae']:.4f} | "
        f"val_loss={row['val_loss']:.6f} | "
        f"val_acc={row['val_acc']:.4f} | "
        f"val_mae={row['val_mae']:.4f} | "
        f"{'BEST' if improved else ''}"
    )

    if improved:
        best_val_loss = row["val_loss"]
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  -> saved best model to {BEST_MODEL_PATH}")
    else:
        epochs_without_improvement += 1

    if USE_EARLY_STOPPING and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}.")
        break

torch.save(model.state_dict(), LAST_MODEL_PATH)
print("\nsaved last model:", LAST_MODEL_PATH)

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_CSV_PATH, index=False)
print("saved history:", HISTORY_CSV_PATH)

print("\nBest epoch:", best_epoch)
print("Best val loss:", best_val_loss)

display(history_df.tail())

In [ ]:
best_model = ValueResNet(
    in_channels=INPUT_CHANNELS,
    trunk_channels=TRUNK_CHANNELS,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
    head_hidden=HEAD_HIDDEN
).to(device)

best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
best_model.eval()

best_val = eval_one_epoch(best_model, val_loader, criterion)

print(
    f"\nBest model | "
    f"val_loss={best_val['loss']:.6f} | "
    f"val_acc={best_val['acc']:.4f} | "
    f"val_mae={best_val['mae']:.4f} | "
    f"val_rmse={best_val['rmse']:.4f}"
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Huber loss")
plt.title("Train / Val Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
plt.plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
plt.xlabel("Epoch")
plt.ylabel(f"Accuracy (|pred-target| <= {ACC_TOLERANCE})")
plt.title("Train / Val Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_mae"], label="train_mae")
plt.plot(history_df["epoch"], history_df["val_mae"], label="val_mae")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.title("Train / Val MAE")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(best_val["targets"], bins=60, alpha=0.6, label="target")
plt.hist(best_val["preds"], bins=60, alpha=0.6, label="pred")
plt.xlabel("Normalized value")
plt.ylabel("Count")
plt.title("Val Target vs Prediction Distribution")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(best_val["targets"], best_val["preds"], s=6, alpha=0.2)
mn = min(best_val["targets"].min(), best_val["preds"].min())
mx = max(best_val["targets"].max(), best_val["preds"].max())
plt.plot([mn, mx], [mn, mx], linestyle="--")
plt.xlabel("Target")
plt.ylabel("Prediction")
plt.title("Val Prediction vs Target")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import onnx
ONNX_PATH = OUTPUT_DIR / "new_value_network.onnx"

dummy_input = torch.randn(1, INPUT_CHANNELS, BOARD_H, BOARD_W, device=device)

torch.onnx.export(
    best_model,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"},
    }
)

print(f"Exported ONNX to {ONNX_PATH}")